# LLM-as-router: can a language model do the routing itself?

The last of the instructor's three suggestions, and the only one that needed genuinely
new work: instead of a trained classifier, hand each request to an LLM together with
the model menu and let it choose. I used gemini-2.5-flash-lite as the routing model
(temperature 0, thinking turned off) because the router itself has to be cheap for the
idea to make any economic sense.

Setup decisions. The router sees the same information regime the trained models had:
the request text plus a menu built from training data only (each model's average cost
per request and the share of training prompts it solved). Request text is truncated to
1,500 characters, a router deciding where to send a request should not need to read a
20-page diff in full. I ran a stratified sample of 300 test prompts rather than all
2,434: the API bill stays at a few cents inside the free tier, the comparison needs matched prompts,
not volume, and every other strategy is re-scored on the identical sample so the
points stay comparable. Responses are cached to a parquet, so re-running this notebook
does not call the API again.

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

proc = Path("..") / "data" / "processed"
long_df = pd.read_parquet(proc / "records_long.parquet")
core = long_df[~long_df.is_reference]
openrouter = long_df[long_df.is_reference]
model_df = pd.read_parquet(proc / "prompts_features_labels_tau100.parquet")
queries = pd.read_parquet(proc / "queries.parquet").set_index("prompt_id")["query"]
router_preds = pd.read_parquet(proc / "router_test_predictions.parquet")
reg_picks = pd.read_parquet(proc / "regressor_test_picks.parquet")

train_df, test_df = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df.dataset)

# stratified 300-prompt sample of the test set
sample_df, _ = train_test_split(
    test_df, train_size=300, random_state=42, stratify=test_df.dataset)
print("sample by dataset:")
print(sample_df.dataset.value_counts())

In [ ]:
# the menu the router reads: built from TRAIN prompts only
train_rows = core[core.prompt_id.isin(train_df.prompt_id)]
menu_stats = train_rows.groupby("model").agg(
    avg_cost=("cost", "mean"), solve_rate=("score", lambda s: (s >= 1.0).mean()))
menu_stats = menu_stats.sort_values("avg_cost")
menu_text = "\n".join(
    f"- {m}: avg cost ${r.avg_cost:.4f} per request, solves {r.solve_rate:.0%} of requests"
    for m, r in menu_stats.iterrows())
print(menu_text)

In [ ]:
ROUTER_MODEL = "gemini-2.5-flash-lite"  # the router must cost less than the savings it finds
TRUNCATE = 1500
CACHE = proc / "llm_router_picks.parquet"

PROMPT_TEMPLATE = (
    "You route user requests to the most economical AI model. Below is a menu of "
    "models with their average cost per request in USD and the share of requests "
    "each solves correctly, then one user request. Pick the single cheapest model "
    "you expect to solve THIS request correctly. Reply with the model name only, "
    "nothing else.\n\nMENU:\n{menu}\n\nREQUEST (may be truncated):\n{request}"
)

valid_names = set(menu_stats.index)
cheapest_model = menu_stats.avg_cost.idxmin()

if CACHE.exists():
    llm_picks_df = pd.read_parquet(CACHE)
    print(f"loaded {len(llm_picks_df)} cached picks, no API calls made")
else:
    from google import genai
    from google.genai import types

    load_dotenv(Path("..") / ".env")
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY")
                          or os.getenv("GOOGLE_API_KEY"))
    config = types.GenerateContentConfig(
        temperature=0, max_output_tokens=50,
        thinking_config=types.ThinkingConfig(thinking_budget=0))

    def ask_router(prompt_id):
        text = PROMPT_TEMPLATE.format(
            menu=menu_text, request=queries.loc[prompt_id][:TRUNCATE])
        for attempt in range(4):
            try:
                resp = client.models.generate_content(
                    model=ROUTER_MODEL, contents=text, config=config)
                answer = (resp.text or "").strip().strip('"').strip("`")
                if answer in valid_names:
                    return prompt_id, answer, "ok"
                return prompt_id, answer, "invalid"
            except Exception:
                time.sleep(5 * (attempt + 1))
        return prompt_id, None, "failed"

    # two workers keeps the call rate inside the API free-tier limits
    with ThreadPoolExecutor(max_workers=2) as pool:
        results = list(pool.map(ask_router, sample_df.prompt_id))
    llm_picks_df = pd.DataFrame(results, columns=["prompt_id", "raw", "status"])
    llm_picks_df["pick"] = llm_picks_df.apply(
        lambda r: r.raw if r.status == "ok" else cheapest_model, axis=1)
    llm_picks_df.to_parquet(CACHE, index=False)
    print("saved picks to cache")

print(llm_picks_df.status.value_counts().to_dict())
print("\ntraffic shares chosen by the LLM router:")
print(llm_picks_df.pick.value_counts(normalize=True).round(3).head(8))

*(Interpretation pending: the free-tier Gemini quota ran out mid-experiment on Jul 21. The call cell caches results, so once quota is available again, one clean execution of this notebook completes the experiment and this cell gets written from the real numbers.)*

In [ ]:
# every strategy re-scored on the identical 300 prompts
sample_rows = core[core.prompt_id.isin(sample_df.prompt_id)]
lookup = sample_rows.set_index(["prompt_id", "model"])[["cost", "score"]]

def outcome(picks):
    got = lookup.loc[list(zip(picks.index, picks.values, strict=True))]
    return got.cost.mean(), got.score.mean()

fixed = sample_rows.groupby("model").agg(cost=("cost", "mean"),
                                         score=("score", "mean"))
sid = sample_df.prompt_id
strategies = {
    "always cheapest": pd.Series(fixed.cost.idxmin(), index=sid),
    "always strongest": pd.Series(fixed.score.idxmax(), index=sid),
    "logistic router (tuned)": router_preds.set_index("prompt_id").logistic_tuned.loc[sid],
    "random forest router (tuned)": router_preds.set_index(
        "prompt_id").random_forest_tuned.loc[sid],
    "regressor-derived router": reg_picks.set_index("prompt_id").regressor_routing.loc[sid],
    "LLM-as-router (gemini-2.5-flash-lite)": llm_picks_df.set_index("prompt_id")["pick"].loc[sid],
    "oracle label": sample_df.set_index("prompt_id").label,
}
rows = [{"strategy": k, "mean cost (USD)": outcome(p)[0], "mean score": outcome(p)[1]}
        for k, p in strategies.items()]
or_sample = openrouter[openrouter.prompt_id.isin(sid)]
rows.append({"strategy": "OpenRouter (reference)",
             "mean cost (USD)": or_sample.cost.mean(),
             "mean score": or_sample.score.mean()})
scoreboard = pd.DataFrame(rows).set_index("strategy")
scoreboard.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(fixed.cost, fixed.score, color="lightgray", s=35, zorder=1,
           label="each model as a fixed strategy")
for name, row in scoreboard.iterrows():
    emph = "LLM-as-router" in name
    ax.scatter(row["mean cost (USD)"], row["mean score"], s=140 if emph else 90,
               zorder=3 if emph else 2, marker="*" if emph else "o")
    ax.annotate(name, (row["mean cost (USD)"], row["mean score"]), fontsize=8,
                xytext=(6, 4), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("mean cost per prompt (USD, log scale)")
ax.set_ylabel("mean score")
ax.set_title("All strategies including the LLM router, 300 sampled test prompts")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

*(Interpretation pending: the free-tier Gemini quota ran out mid-experiment on Jul 21. The call cell caches results, so once quota is available again, one clean execution of this notebook completes the experiment and this cell gets written from the real numbers.)*